<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-3-rag-pipelines/Module_3_Session_3_Full_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 - Session 3: Full RAG Pipeline
Goal: Load a Swiggy policy PDF, chunk it, embed it, store in FAISS, and answer questions from it.

In [2]:
# Install all libraries needed for this session
# sentence-transformers — converts text to vectors
# faiss-cpu — vector database to store and search embeddings
# pymupdf — reads PDF files (you used this in Module 2 Session 2)
# groq — LLM to generate the final answer
!pip install -q sentence-transformers faiss-cpu pymupdf groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.2 MB/s eta 0:00:00


## Step 2 - Create Swiggy Delivery Partner Policy PDF
We create a real multi-section PDF document.
This is our knowledge base — the document our RAG pipeline will search.

In [3]:
# Import fitz — this is pymupdf's main module
# You used this in Module 2 Session 2 to read PDFs
# Today we use it to CREATE a PDF
import fitz

# Create a new empty PDF document
doc = fitz.open()

# Our Swiggy policy content — multiple sections
# Each section covers a different topic
# This simulates a real enterprise policy document
policy_sections = [
    {
        "title": "Section 1: Delivery Partner Eligibility",
        "content": (
            "To become a Swiggy delivery partner, you must be at least 18 years old. "
            "You must own a smartphone with Android 6.0 or above. "
            "A valid driving license and vehicle registration certificate are mandatory. "
            "Two-wheeler or bicycle ownership is required for deliveries. "
            "You must have a valid Aadhaar card and PAN card for verification. "
            "Background verification will be conducted before onboarding. "
            "Partners must complete the Swiggy training module before first delivery."
        )
    },
    {
        "title": "Section 2: Earnings and Payments",
        "content": (
            "Delivery partners earn a base fee for every order delivered. "
            "Additional incentives are provided for completing surge orders during peak hours. "
            "Weekly payments are credited directly to the registered bank account. "
            "Partners can track their daily and weekly earnings in the Swiggy Partner app. "
            "Surge bonuses are calculated based on orders completed in high demand zones. "
            "Tips provided by customers are credited in full to the delivery partner. "
            "Payment disputes must be raised within 7 days through the partner support portal."
        )
    },
    {
        "title": "Section 3: Order Handling and Delivery Rules",
        "content": (
            "Delivery partners must pick up orders within 5 minutes of reaching the restaurant. "
            "Orders must be delivered within the estimated time shown in the app. "
            "Partners must not open or tamper with food packaging at any time. "
            "If a customer is unreachable, partners must wait 5 minutes before contacting support. "
            "Proof of delivery photo must be uploaded for all orders above Rs 500. "
            "Partners must follow the route suggested by the Swiggy navigation system. "
            "Any delivery delay beyond 15 minutes must be reported through the app."
        )
    },
    {
        "title": "Section 4: Code of Conduct",
        "content": (
            "Delivery partners must maintain professional behavior with customers at all times. "
            "Use of abusive language or aggressive behavior will result in immediate suspension. "
            "Partners must wear the Swiggy delivery uniform and helmet during all deliveries. "
            "Mobile phones must not be used while riding the delivery vehicle. "
            "Partners must not accept orders outside the Swiggy platform. "
            "Any form of fraud including fake deliveries will result in permanent deactivation. "
            "Partners must report any accident or incident immediately through the app."
        )
    },
    {
        "title": "Section 5: Grievance and Support",
        "content": (
            "Delivery partners can raise grievances through the Swiggy Partner app support section. "
            "All grievances will be acknowledged within 24 hours of submission. "
            "Resolution of grievances will be provided within 5 working days. "
            "Partners can escalate unresolved issues to the regional partner support team. "
            "Swiggy provides accident insurance coverage for all active delivery partners. "
            "Medical emergencies during delivery must be reported within 48 hours. "
            "Partners have the right to appeal any suspension or deactivation decision."
        )
    }
]

# Add each section as a page in the PDF
for section in policy_sections:
    # Add a blank page
    page = doc.new_page()

    # Insert the title at the top
    page.insert_text(
        (50, 50),
        section["title"],
        fontsize=14,
        color=(0, 0, 0)
    )

    # Insert the content below the title
    page.insert_text(
        (50, 100),
        section["content"],
        fontsize=10,
        color=(0, 0, 0)
    )

# Save the PDF to Colab's local storage
pdf_path = "/content/swiggy_delivery_policy.pdf"
doc.save(pdf_path)
doc.close()

print(f"PDF created successfully at: {pdf_path}")
print(f"Total sections: {len(policy_sections)}")

PDF created successfully at: /content/swiggy_delivery_policy.pdf
Total sections: 5


## Step 3 - Extract Text and Split into Chunks
We extract text from each PDF page and split into smaller chunks.
Chunking is critical — if chunks are too big the LLM gets confused,
if chunks are too small they lose context.

In [4]:
import fitz

# Open the PDF we just created
doc = fitz.open(pdf_path)

# Extract text from each page
# Each page is one policy section in our case
chunks = []

for page_num in range(len(doc)):
    # Extract text from this page
    page = doc[page_num]
    text = page.get_text()

    # Clean up extra whitespace
    text = text.strip()

    # Only add if page has actual content
    if text:
        chunks.append({
            "page": page_num + 1,        # which page this came from
            "text": text                  # the actual text content
        })

doc.close()

# Print what we extracted
print(f"Total chunks extracted: {len(chunks)}")
print(f"\n--- Preview of Chunk 1 ---")
print(f"Page: {chunks[0]['page']}")
print(f"Text: {chunks[0]['text'][:200]}...")

Total chunks extracted: 5

--- Preview of Chunk 1 ---
Page: 1
Text: Section 1: Delivery Partner Eligibility
To become a Swiggy delivery partner, you must be at least 18 years old. You must own a smartphone with Android 6.0 or a...


## Step 4 - Embed Chunks and Store in FAISS
This is the OFFLINE step — happens once when we set up the pipeline.
Every chunk becomes a vector and gets stored in FAISS.

In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load the same embedding model we used in Sessions 1 and 2
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract just the text from each chunk for embedding
chunk_texts = [chunk["text"] for chunk in chunks]

# Convert all 5 chunks into embeddings
# Shape will be (5, 384) — 5 chunks, 384 numbers each
chunk_embeddings = model.encode(chunk_texts)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

# Convert to float32 — FAISS requirement
chunk_embeddings_float = np.array(chunk_embeddings).astype("float32")

# Normalize for cosine similarity — same as Session 2
faiss.normalize_L2(chunk_embeddings_float)

# Build FAISS index with cosine similarity
index = faiss.IndexFlatIP(384)

# Add all chunk embeddings to FAISS
index.add(chunk_embeddings_float)

print(f"FAISS index built successfully!")
print(f"Total chunks stored in FAISS: {index.ntotal}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunk embeddings shape: (5, 384)
FAISS index built successfully!
Total chunks stored in FAISS: 5


## Step 5 - Search FAISS with a Real Question
This is the ONLINE step — happens every time someone asks a question.
We convert the question to a vector and find the most relevant policy chunk.

In [6]:
# A Swiggy delivery partner asks a question
partner_question = "How much time do I have to raise a payment dispute?"

# Convert question to embedding — same model, same 384 numbers
question_embedding = model.encode([partner_question])

# Convert to float32 and normalize — must match how we stored the chunks
question_embedding_float = np.array(question_embedding).astype("float32")
faiss.normalize_L2(question_embedding_float)

# Search FAISS for top 2 most relevant chunks
distances, indices = index.search(question_embedding_float, k=2)

# Print retrieved chunks
print(f"Question: '{partner_question}'\n")
print("Top 2 retrieved chunks:")
print("-" * 50)

for rank, (score, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"\nRank {rank+1} — Score: {score:.4f}")
    print(f"Page: {chunks[idx]['page']}")
    print(f"Text preview: {chunks[idx]['text'][:150]}...")

Question: 'How much time do I have to raise a payment dispute?'

Top 2 retrieved chunks:
--------------------------------------------------

Rank 1 — Score: 0.2830
Page: 3
Text preview: Section 3: Order Handling and Delivery Rules
Delivery partners must pick up orders within 5 minutes of reaching the restaurant. Orders must be deliver...

Rank 2 — Score: 0.2484
Page: 5
Text preview: Section 5: Grievance and Support
Delivery partners can raise grievances through the Swiggy Partner app support section. All grievances will be acknowl...


## Step 6 - Send Retrieved Chunks to LLM
Even if retrieval is imperfect, the LLM tries to reason from what it got.
We will see if it finds the right answer from the retrieved context.

In [7]:
import os
from groq import Groq
from google.colab import userdata

# Load Groq API key
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# Collect top 2 retrieved chunks as context
retrieved_chunks = [chunks[idx]["text"] for idx in indices[0][:2]]

# Build context string
context = "\n\n---\n\n".join(retrieved_chunks)

# Build the prompt
prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy document sections to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information in the policy."

Policy Context:
{context}

Delivery Partner Question: {partner_question}

Answer clearly and directly in 2-3 sentences."""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print(f"Question: '{partner_question}'\n")
print(f"LLM Answer:")
print(response.choices[0].message.content)

Question: 'How much time do I have to raise a payment dispute?'

LLM Answer:
I could not find this information in the policy.


## Step 7 - Test with Different Questions
We test the full pipeline with questions that map clearly to each section.
This shows us which parts of our knowledge base retrieve well.

In [8]:
# Let's test with 3 different questions — one per section
# This shows the pipeline working end to end correctly
test_questions = [
    "What documents do I need to become a delivery partner?",
    "When do I get paid and how are earnings calculated?",
    "What happens if a customer is not reachable at delivery?"
]

for question in test_questions:
    # Embed the question
    q_embedding = model.encode([question])
    q_embedding_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_embedding_float)

    # Search FAISS
    distances, indices = index.search(q_embedding_float, k=1)

    # Get top chunk
    top_chunk = chunks[indices[0][0]]["text"]
    top_score = distances[0][0]
    top_page = chunks[indices[0][0]]["page"]

    # Send to LLM
    prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy section to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information."

Policy Context:
{top_chunk}

Question: {question}

Answer clearly in 2 sentences."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    print(f"Question: '{question}'")
    print(f"Retrieved: Page {top_page} — Score: {top_score:.4f}")
    print(f"Answer: {response.choices[0].message.content}")
    print("-" * 60)

Question: 'What documents do I need to become a delivery partner?'
Retrieved: Page 1 — Score: 0.5024
Answer: I could not find this information. The policy only mentions that you must own a smartphone with Android 6.0 or a higher version, but it does not mention what documents are required to become a delivery partner.
------------------------------------------------------------
Question: 'When do I get paid and how are earnings calculated?'
Retrieved: Page 2 — Score: 0.3118
Answer: As a Swiggy delivery partner, you get paid every 7 days, and your earnings are calculated based on the base fee for every order delivered, with additional incentives provided for surge orders and other specific conditions. 

However, as per the policy context provided, there's no clear information about the detailed calculation process of earnings or how incentives for surge orders specifically work. To get a precise answer, I would require more information or the full policy context.
-----------------------

## Step 8 - Fix PDF Text Truncation
pymupdf's insert_text cuts off long text.
We use insert_textbox instead which wraps text properly within a defined area.

In [9]:
import fitz

# Recreate the PDF with insert_textbox instead of insert_text
# insert_textbox takes a rectangle area and wraps text inside it
doc = fitz.open()

for section in policy_sections:
    page = doc.new_page()

    # Insert title at top
    page.insert_text(
        (50, 50),
        section["title"],
        fontsize=14,
        color=(0, 0, 0)
    )

    # Define a rectangle for the content area
    # (x0, y0, x1, y1) — starts at 50,80 and fills most of the page
    rect = fitz.Rect(50, 80, 550, 750)

    # insert_textbox wraps text inside the rectangle
    # It never truncates — all content fits properly
    page.insert_textbox(
        rect,
        section["content"],
        fontsize=10,
        color=(0, 0, 0)
    )

doc.save(pdf_path)
doc.close()

print("PDF recreated with proper text wrapping!")

# Verify by reading it back
doc = fitz.open(pdf_path)
for i in range(len(doc)):
    page = doc[i]
    text = page.get_text().strip()
    print(f"Page {i+1} — {len(text)} characters extracted")
doc.close()

PDF recreated with proper text wrapping!
Page 1 — 501 characters extracted
Page 2 — 555 characters extracted
Page 3 — 563 characters extracted
Page 4 — 559 characters extracted
Page 5 — 552 characters extracted


## Step 9 - Rebuild Full Pipeline with Fixed PDF
Now that PDF text is complete, we re-extract, re-embed, and rebuild FAISS.
Then we test all 3 questions again to see if answers improve.

In [10]:
# STEP A — Re-extract chunks from fixed PDF
doc = fitz.open(pdf_path)
chunks = []

for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text().strip()
    if text:
        chunks.append({
            "page": page_num + 1,
            "text": text
        })

doc.close()
print(f"Chunks re-extracted: {len(chunks)}")

# STEP B — Re-embed all chunks
chunk_texts = [chunk["text"] for chunk in chunks]
chunk_embeddings = model.encode(chunk_texts)
chunk_embeddings_float = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(chunk_embeddings_float)

# STEP C — Rebuild FAISS index
index = faiss.IndexFlatIP(384)
index.add(chunk_embeddings_float)
print(f"FAISS rebuilt with {index.ntotal} chunks")

# STEP D — Test all 3 questions again
test_questions = [
    "What documents do I need to become a delivery partner?",
    "When do I get paid and how are earnings calculated?",
    "What happens if a customer is not reachable at delivery?"
]

print("\n" + "=" * 60)

for question in test_questions:
    # Embed question
    q_embedding = model.encode([question])
    q_embedding_float = np.array(q_embedding).astype("float32")
    faiss.normalize_L2(q_embedding_float)

    # Search FAISS
    distances, indices = index.search(q_embedding_float, k=1)

    # Get top chunk
    top_chunk = chunks[indices[0][0]]["text"]
    top_score = distances[0][0]
    top_page = chunks[indices[0][0]]["page"]

    # Send to LLM
    prompt = f"""You are a Swiggy delivery partner support agent.

Use the following policy section to answer the question.
Only use the context provided — do not make up information.
If the answer is not in the context, say "I could not find this information."

Policy Context:
{top_chunk}

Question: {question}

Answer clearly in 2 sentences."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    print(f"\nQuestion: '{question}'")
    print(f"Retrieved: Page {top_page} — Score: {top_score:.4f}")
    print(f"Answer: {response.choices[0].message.content}")
    print("-" * 60)

Chunks re-extracted: 5
FAISS rebuilt with 5 chunks


Question: 'What documents do I need to become a delivery partner?'
Retrieved: Page 1 — Score: 0.5672
Answer: To become a Swiggy delivery partner, you will need to have a valid driving license and vehicle registration certificate. Additionally, you must also have a valid Aadhaar card and PAN card for verification.
------------------------------------------------------------

Question: 'When do I get paid and how are earnings calculated?'
Retrieved: Page 2 — Score: 0.3926
Answer: As a Swiggy delivery partner, you earn a base fee for every order delivered, and additional incentives like surge bonuses for completing surge orders during peak hours. Your weekly payments are credited directly to your registered bank account, and you can track your daily and weekly earnings in the Swiggy Partner app. 

(I combined both parts of your request into one clear response)
------------------------------------------------------------

Question: 'What

## What We Built
A complete end-to-end RAG pipeline:
- Created a 5-section Swiggy delivery partner policy PDF
- Extracted text from each page using pymupdf
- Split document into chunks — one chunk per page
- Embedded all chunks using sentence-transformers (OFFLINE step)
- Stored chunk embeddings in FAISS with cosine similarity (OFFLINE step)
- Took delivery partner questions, embedded them (ONLINE step)
- Retrieved top matching chunks from FAISS (ONLINE step)
- Sent retrieved chunks + question to Groq LLM for grounded answers
- Learned that PDF text quality directly impacts RAG answer quality

## Key Lessons Learned
- RAG pipeline = Retrieve then Generate — LLM cannot fix bad retrieval
- Always verify source document quality before debugging the algorithm
- insert_textbox wraps text properly — insert_text truncates long content
- Grounding prompt ("if not in context, say so") prevents hallucination
- Retrieval scores improved when document content was complete

## FAANG Interview Answer
"A RAG pipeline is only as good as its retrieval.
If retrieval fails, even the best LLM cannot give a correct answer.
Document quality, chunk size, and embedding model choice are the
three biggest factors in RAG retrieval quality."

## AWS Equivalent
| What we did here         | AWS service                          |
|--------------------------|--------------------------------------|
| pymupdf PDF extraction   | Amazon Textract                      |
| Chunking by page         | Amazon Bedrock Knowledge Bases       |
| sentence-transformers    | Amazon Titan Embeddings              |
| FAISS vector store       | Amazon OpenSearch Serverless         |
| Groq LLM                 | Amazon Bedrock (Claude/Llama)        |
| Full RAG pipeline        | Amazon Bedrock Knowledge Bases       |

## What is Next
Session 4 — Chunking strategies and retrieval quality.
We learn why chunk size matters and how to split documents smartly
so retrieval scores go above 0.6 consistently.